# Milestone - Unit 12: sequence-aware retrieval (what you read *last* matters) -- Every path in Units 2-11 treats a reader's history as a **bag**: the scoreboard hands each path an unordered `seen` set. This milestone scores three **order-aware** paths on the separate **session log**, where the order of reads carries signal (series next-volumes, author streaks, genre moods): **last-k item-item CF** (Unit 4's cosine, queried with only the last 3 distinct reads), a first-order **item transition** model ("what do people read right after X?"), and a tiny **SASRec-style causal self-attention** model at the frozen fast recipe D. The honest, measured story: the two cheap order-aware paths carry the lift (last-3 ~0.551 and transition ~0.516 vs bag item-item ~0.480); **order-shuffled controls** prove that *order*, not the item set, carries it (transition collapses to ~0.333; SASRec drops from ~0.423 to ~0.383, about 3.8 paired standard errors); and SASRec at this recipe **trails** plain item-item CF by about 0.06 -- it learns order, but it does not win here. Everything is fixed-seed and deterministic, and we never touch `split="test"`.

## 1. Load the session log

Unit 12 runs on **new data**: `sessions.csv.gz`, a separate seeded session log with the same columns as the main interaction log plus a `session_id`. It has a **different reader population**, so every number below is "on the session log" and is **not comparable** to the main-log numbers of Units 2-11. The book catalog is the same. `series.csv.gz` (which books are volumes of which series) is used **only as an evaluation lens** in section 5 -- never as a model input. `bookrec.generated_dir()` reports a clear error if `recsys/data/generated/` has not been built.

In [ ]:
from time import perf_counter

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    ItemItemRetrievalPath,
    LastKItemItemPath,
    PathRegistry,
    PopularityRetrievalPath,
    SasrecRetrievalPath,
    TransitionRetrievalPath,
    first_unseen_val,
    load_catalog,
    load_series,
    load_train_sequences,
    rank,
    reader_hits,
    run_blended_scoreboard,
)

K = 10
SHUFFLE_SEED = 20260930 + 1000   # the order-shuffled control seed (committed data seed + 1000)

data_dir = bookrec.generated_dir()
catalog_ids = sorted(load_catalog(data_dir / "catalog.csv.gz"))
sessions_path = data_dir / "sessions.csv.gz"        # the SESSION log -- a different reader population
rows = pd.read_csv(sessions_path).to_dict("records")  # reader_id/item_id/session_id/timestamp/split/label
histories = load_train_sequences(sessions_path)       # each reader's ORDERED train positives
series = load_series(data_dir / "series.csv.gz")      # {book: (series_id, volume)} -- a lens only

{
    "books": len(catalog_ids),
    "session_rows": len(rows),
    "readers_with_history": len(histories),
    "median_history_length": int(np.median([len(items) for items in histories.values()])),
    "books_in_a_series": len(series),
}

## 2. Fit the bag baselines and the cheap order-aware paths

The baselines are the Unit-2 **popularity** path and the Unit-4 bag **item-item** CF. The two cheap order-aware paths are numpy-only and fit in seconds: `LastKItemItemPath(3)` reuses the item-item cosine but queries with the reader's **last 3 distinct** reads, and `TransitionRetrievalPath(5)` counts consecutive train pairs `T[a, b]` and scores the summed rows of the last 5 reads. Every sequence path builds its ordered histories from the train positives only (val/test rows never enter a count or a history), and its `retrieve` queries with that **stored** history -- the scoreboard's `seen` set is only used to exclude books already read.

The **order-shuffled control** is the same transition path with `shuffle_seed` set: each reader's history is permuted (seeded) before learning **and** before querying. The item set is unchanged; only the order is destroyed.

In [ ]:
fit_start = perf_counter()
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
last_3 = LastKItemItemPath(3).fit(rows, catalog=catalog_ids)
transition = TransitionRetrievalPath(5).fit(rows, catalog=catalog_ids)
transition_shuffled = TransitionRetrievalPath(5, shuffle_seed=SHUFFLE_SEED, name="transition-shuffled").fit(
    rows, catalog=catalog_ids
)
numpy_fit_seconds = perf_counter() - fit_start

# The control really permutes: same multiset of books per reader, different order.
example = min(histories)
assert sorted(transition_shuffled.histories[example]) == sorted(transition.histories[example])
assert transition.histories == histories   # the stored ordered history IS the loader's history

print(f"numpy fits (popularity, item-item, last-3, transition, transition-shuffled): {numpy_fit_seconds:.1f}s")
pd.DataFrame({
    "reader": [example] * 6,
    "position": list(range(1, 7)),
    "ordered history": transition.histories[example][:6],
    "shuffled control": transition_shuffled.histories[example][:6],
})

## 3. Train SASRec twice: ordered, then order-shuffled

`SasrecRetrievalPath()` is a tiny causal self-attention model -- 1 block, 1 head, dim 32, the last <= 50 reads, learned positions, a causal mask (each position attends only to earlier reads), tied output embeddings, and per-position next-item cross-entropy. Its defaults are the **frozen recipe D** (30 epochs, batch 128, Adam lr 2e-3, dropout 0.2, seed 0), chosen before the unit was built and never re-tuned. Training is deterministic on one CPU thread; serving runs a numpy forward pass, so `retrieve` never needs torch. Each fit sits in its own cell (~45-60 s each, measured).

In [ ]:
sasrec_start = perf_counter()
sasrec = SasrecRetrievalPath().fit(rows, catalog=catalog_ids)
sasrec_seconds = perf_counter() - sasrec_start

assert sasrec.loss_history[-1] < sasrec.loss_history[0]   # training actually reduced the loss
print(f"SASRec (ordered, recipe D) fit: {sasrec_seconds:.1f}s | loss "
      f"{sasrec.loss_history[0]:.3f} -> {sasrec.loss_history[-1]:.3f} over {len(sasrec.loss_history)} epochs")

The control: the **same** recipe and model seed, but every training sequence (and every query history) is shuffled with the same `SHUFFLE_SEED`. If SASRec's attention is really using *order*, this copy must score lower.

In [ ]:
sasrec_shuffled_start = perf_counter()
sasrec_shuffled = SasrecRetrievalPath(shuffle_seed=SHUFFLE_SEED, name="sasrec-shuffled").fit(
    rows, catalog=catalog_ids
)
sasrec_shuffled_seconds = perf_counter() - sasrec_shuffled_start

print(f"SASRec (order-shuffled control) fit: {sasrec_shuffled_seconds:.1f}s")

Register the three shipped sequence paths. The registry pins each one's name and artifact id (`last-k-item-item-v1`, `transition-v1`, `sasrec-v1`) -- the controls are measurement tools, not shipped paths, so they are not registered.

In [ ]:
registry = PathRegistry()
for path in (last_3, transition, sasrec):
    registry.register(path)

assert [path.artifact_name() for path in (last_3, transition, sasrec)] == [
    "last-k-item-item-v1", "transition-v1", "sasrec-v1",
]
assert registry.get("sasrec") is sasrec
{path.name: path.artifact_name() for path in (last_3, transition, sasrec)}

## 4. The session-log val scoreboard

Three readouts per path, all on val at k = 10:
- **hit@10** -- the unchanged scoreboard cohort (readers with at least one unseen val positive), computed per reader with `reader_hits` so two paths can be **paired** reader by reader;
- **next-item hit@10** -- does the top-10 contain the reader's time-ordered *first* unseen val book (`first_unseen_val`)? This is the sharper "what comes next" question;
- **coverage** -- the fraction of the catalog that appears in at least one reader's top-10 (a single-path `run_blended_scoreboard` with `pool=K`, which reproduces the path's hit@10 exactly).

In [ ]:
paths = {
    "popularity": popularity,
    "item-item (bag)": item_item,
    "last-3 item-item": last_3,
    "transition": transition,
    "transition (shuffled)": transition_shuffled,
    "sasrec": sasrec,
    "sasrec (shuffled)": sasrec_shuffled,
}
next_item = first_unseen_val(sessions_path)


class Remembered:
    """Wrap a fitted path so each reader's top-K is computed once and reused by all three readouts."""

    def __init__(self, path):
        self.path, self.name, self.memo = path, path.name, {}

    def retrieve(self, query, context, k):
        if (query, k) not in self.memo:
            self.memo[(query, k)] = self.path.retrieve(query, context, k)
        return self.memo[(query, k)]


score_start = perf_counter()
hits, board = {}, []
next_item_readers = 0
for label, fitted in paths.items():
    path = Remembered(fitted)   # SASRec's numpy forward pass is the slow part; run it once per reader
    hits[label] = reader_hits(path, sessions_path, catalog_ids=catalog_ids, k=K)
    coverage_run = run_blended_scoreboard({path.name: path}, sessions_path, catalog_ids=catalog_ids, k=K, pool=K)
    hit_at_k = float(np.mean(list(hits[label].values())))
    assert abs(coverage_run.hit_rate_at_k - hit_at_k) < 1e-9   # the two harnesses agree on the cohort
    next_hits = []
    for reader in hits[label]:
        if reader not in next_item:
            continue
        seen = set(histories.get(reader, []))
        top = [c.item_id for c in rank(path.retrieve(reader, {"seen": seen}, K), n=K, exclude=seen)]
        next_hits.append(1.0 if next_item[reader] in top else 0.0)
    next_item_readers = len(next_hits)
    board.append({
        "path": label,
        "hit@10": round(hit_at_k, 4),
        "next-item hit@10": round(float(np.mean(next_hits)), 4),
        "coverage": round(coverage_run.catalog_coverage, 4),
    })
score_seconds = perf_counter() - score_start

cohort = len(hits["item-item (bag)"])
assert all(len(per_reader) == cohort for per_reader in hits.values())   # every path, the same readers
print(f"scored {len(paths)} paths over {cohort} val readers ({next_item_readers} with a next-item target) "
      f"in {score_seconds:.1f}s -- session log, NOT comparable to Units 2-11")
pd.DataFrame(board)

### The gates: recency lift, and order evidence from two controls

A raw hit@10 cannot say *why* a path wins. The order claim is made by **controls**: the same path, the same item sets, order destroyed. For SASRec the drop is smaller, so we also need its noise level -- the **paired standard error** of the per-reader hit difference (both models are scored on the same readers, so pairing removes the reader-to-reader variance). The gates, fixed before the unit was built:
- **(a) recency:** last-3 >= bag item-item + 0.03;
- **(b) transition order:** ordered >= 1.3 x shuffled **and** ordered - shuffled >= 0.10;
- **(c) SASRec learns order:** ordered - shuffled >= 0.02 **and** >= 2 paired standard errors.

In [ ]:
def hit(label):
    return float(np.mean(list(hits[label].values())))


def paired(label_a, label_b):
    """Mean per-reader hit difference (a - b) and its paired standard error."""
    diffs = np.array([hits[label_a][reader] - hits[label_b][reader] for reader in sorted(hits[label_a])])
    return float(diffs.mean()), float(diffs.std(ddof=1) / np.sqrt(len(diffs)))


recency_lift, recency_se = paired("last-3 item-item", "item-item (bag)")
transition_gap, transition_se = paired("transition", "transition (shuffled)")
sasrec_order, sasrec_order_se = paired("sasrec", "sasrec (shuffled)")
sasrec_vs_bag, sasrec_vs_bag_se = paired("sasrec", "item-item (bag)")

# GATE (a): recency -- querying with the last 3 distinct reads beats the whole bag.
assert recency_lift >= 0.03
# GATE (b): order evidence -- shuffling the transition path's sequences collapses its lift.
assert hit("transition") >= 1.3 * hit("transition (shuffled)")
assert transition_gap >= 0.10
# GATE (c): SASRec learns order -- it beats its own order-shuffled copy, clearly outside the noise.
assert sasrec_order >= 0.02
assert sasrec_order >= 2 * sasrec_order_se

pd.DataFrame([
    {"comparison": "(a) last-3 - bag item-item", "difference": round(recency_lift, 4),
     "paired SE": round(recency_se, 4), "gated": "yes"},
    {"comparison": "(b) transition - transition (shuffled)", "difference": round(transition_gap, 4),
     "paired SE": round(transition_se, 4), "gated": "yes"},
    {"comparison": "(c) sasrec - sasrec (shuffled)", "difference": round(sasrec_order, 4),
     "paired SE": round(sasrec_order_se, 4), "gated": "yes"},
    {"comparison": "sasrec - bag item-item (recorded)", "difference": round(sasrec_vs_bag, 4),
     "paired SE": round(sasrec_vs_bag_se, 4), "gated": "no"},
])

### The honest readout: SASRec learns order, but it does not win here

The last row is **recorded, not gated**. At recipe D, SASRec **trails** bag item-item CF by about 0.06 on the committed seed (measured: about -0.057, roughly 5 paired standard errors -- a real gap, not a tie). The near-tie reported when the session log was built belongs to a slower ~5-minute-per-fit recipe on held-out seeds; it is not this recipe's result, so we do not borrow it. The winners on this data are the cheap paths: the order effects here are **short-range and item-to-item** (the next volume, the same author), which a counting model captures directly, while a tiny attention model at our size and time budget cannot turn its extra flexibility into accuracy.

In [ ]:
ratio = hit("transition") / hit("transition (shuffled)")
print(f"SASRec - bag item-item = {sasrec_vs_bag:+.4f} (paired SE {sasrec_vs_bag_se:.4f}) -- recorded, not gated")
print(f"SASRec order evidence  = {sasrec_order:+.4f} = {sasrec_order / sasrec_order_se:.1f} paired SE")
print(f"transition ordered / shuffled = {ratio:.2f}x")
{"sasrec_fit_s": round(sasrec_seconds, 1), "sasrec_shuffled_fit_s": round(sasrec_shuffled_seconds, 1)}

## 5. One reader: the next volume of a series

Concretely: the first val reader (in id order) whose **last** train read is volume *v* of a series, whose held-out val books include volume *v + 1*, and for whom the transition path puts that next volume in its top-10 while bag item-item CF misses it. `series.csv.gz` is used here only to *find and label* such a reader -- no path ever saw it. The bag path scores the reader's whole history at once, so one recent volume is drowned out by everything else they have read; the transition path asks "what do readers read right after these last 5 books?", and the next volume is the answer.

In [ ]:
volume_of = {(series_id, volume): book for book, (series_id, volume) in series.items()}
val_books = {reader: set() for reader in histories}
for row in rows:
    if row["split"] == "val" and row["label"] == 1 and row["reader_id"] in val_books:
        val_books[row["reader_id"]].add(int(row["item_id"]))


def top_10(path, reader, seen):
    return [c.item_id for c in rank(path.retrieve(reader, {"seen": seen}, K), n=K, exclude=seen)]


found = None
for reader in sorted(hits["transition"]):
    last_book = histories[reader][-1]
    if last_book not in series:
        continue
    series_id, volume = series[last_book]
    next_book = volume_of.get((series_id, volume + 1))
    seen = set(histories[reader])
    if next_book is None or next_book in seen or next_book not in val_books.get(reader, set()):
        continue
    transition_recs = top_10(transition, reader, seen)
    bag_recs = top_10(item_item, reader, seen)
    if next_book in transition_recs and next_book not in bag_recs:
        found = (reader, last_book, next_book, transition_recs, bag_recs)
        break

assert found is not None
reader, last_book, next_book, transition_recs, bag_recs = found


def label(book):
    if book in series:
        return f"{book} (series {series[book][0]}, vol {series[book][1]})"
    return str(book)


print(f"reader {reader}: last 5 reads (oldest -> newest): {[label(b) for b in histories[reader][-5:]]}")
print(f"held-out next volume {label(next_book)}: transition rank "
      f"{next(j for j, book in enumerate(transition_recs, 1) if book == next_book)} of 10; bag item-item misses it")
pd.DataFrame({
    "bag item-item top-10": [f"{label(b)}{'  <-- next volume' if b == next_book else ''}" for b in bag_recs],
    "transition top-10": [f"{label(b)}{'  <-- next volume' if b == next_book else ''}" for b in transition_recs],
})

**The honest headline.** On the session log, order matters -- and the cheap paths capture it. Querying item-item CF with the **last 3** distinct reads lifts hit@10 from ~0.480 to ~0.551; the first-order **transition** path reaches ~0.516, and its **order-shuffled control** collapses to ~0.333, which is what proves that *order*, not the set of books, carries the lift. The tiny **SASRec** attention model also learns order (ordered ~0.423 vs shuffled ~0.383, about +0.040 or 3.8 paired standard errors -- asserted above), but at the fast recipe D it **trails** bag item-item CF by about 0.06. That is the readout we report; it is not a tie. Popularity stays far behind (~0.11), and every number here is on the session log, not comparable to Units 2-11.

**The bridge.** Unit 13 steps back from accuracy and evaluates the whole system -- coverage, fairness and feedback loops -- and Checkpoint B finally opens the sealed `test` holdout; Unit 14's capstone wires retrieval -> blend -> rerank end to end.